In [1]:
import rasterio
import numpy as np
import pandas as pd
import os

os.makedirs('outputs', exist_ok=True)

In [2]:
BASE = r'D:\INTERN - LUNAR SOUTH POLE\Intern_3_granularity_science_priority\raster_layers'

FILES = [
    # filename                                  instrument   role
    ('dfsar_cpr_mosaic_circle.tif',             'DFSAR',    'science_value'),
    ('dfsar_cpr_mosaic_coverage.tif',           'DFSAR',    'confidence'),
    ('gravil_anom_l900_native_10km_circular.tif','GRAIL',   'context'),
    ('gravil_boug_l900_native_10km_circular.tif','GRAIL',   'context'),
    ('lola_downsampled_25m.tif',                'LOLA',     'accessibility'),
    ('lola_downsampled_59m.tif',                'LOLA',     'accessibility'),
    ('lola_downsampled_236m.tif',               'LOLA',     'accessibility'),
    ('lola_matched_to_lroc_5m_clean.tif',       'LOLA',     'accessibility'),
    ('lroc_5m_float32_clean.tif',               'LROC',     'context'),
    ('lroc_shadow_mask_final.tif',              'LROC',     'risk'),
    ('m3_mosaic_group2_july13_14_allbands_280m.tif', 'M3', 'context'),
    ('minirf_s1_pju_mosaic_circle.tif',         'Mini-RF',  'science_value'),
    ('minirf_s1_pju_mosaic_coverage.tif',       'Mini-RF',  'confidence'),
]

In [3]:
rows = []

for fname, instrument, role in FILES:
    fpath = os.path.join(BASE, fname)
    try:
        with rasterio.open(fpath) as src:
            data = src.read(1).astype(float)
            nd   = src.nodata
            mask = (data != nd) if nd is not None else np.isfinite(data)
            valid = data[mask]

            rows.append({
                'filename'    : fname,
                'instrument'  : instrument,
                'role'        : role,
                'width_px'    : src.width,
                'height_px'   : src.height,
                'bands'       : src.count,
                'res_x_m'     : round(src.res[0], 2),
                'res_y_m'     : round(src.res[1], 2),
                'nodata'      : nd,
                'min'         : round(float(valid.min()), 4) if valid.size else 'N/A',
                'max'         : round(float(valid.max()), 4) if valid.size else 'N/A',
                'mean'        : round(float(valid.mean()), 4) if valid.size else 'N/A',
                'status'      : 'OK'
            })
            print(f'OK   {fname}')

    except Exception as e:
        rows.append({'filename': fname, 'instrument': instrument,
                     'role': role, 'status': str(e)})
        print(f'ERR  {fname} -> {e}')

OK   dfsar_cpr_mosaic_circle.tif
OK   dfsar_cpr_mosaic_coverage.tif
OK   gravil_anom_l900_native_10km_circular.tif
OK   gravil_boug_l900_native_10km_circular.tif
OK   lola_downsampled_25m.tif
OK   lola_downsampled_59m.tif
OK   lola_downsampled_236m.tif
OK   lola_matched_to_lroc_5m_clean.tif
OK   lroc_5m_float32_clean.tif
OK   lroc_shadow_mask_final.tif
OK   m3_mosaic_group2_july13_14_allbands_280m.tif
OK   minirf_s1_pju_mosaic_circle.tif
OK   minirf_s1_pju_mosaic_coverage.tif


In [6]:
import os
import pandas as pd

os.makedirs("outputs", exist_ok=True)

df = pd.DataFrame(rows)

df.to_csv("outputs/file_description_table.csv", index=False)

print("Saved to outputs/file_description_table.csv")

df

Saved to outputs/file_description_table.csv


,filename,instrument,role,width_px,height_px,bands,res_x_m,res_y_m,nodata,min,max,mean,status
0,dfsar_cpr_mosaic_circle.tif,DFSAR,science_value,3639,3639,1,25.00,25.00,-9.999000e+03,0.0078,2.9259,0.2982,OK
1,dfsar_cpr_mosaic_coverage.tif,DFSAR,confidence,3639,3639,1,25.00,25.00,2.550000e+02,0.0000,1.0000,0.5878,OK
2,gravil_anom_l900_native_10km_circular.tif,GRAIL,context,9,9,1,10000.00,10000.00,-3.276700e+04,-241.7927,268.3000,52.3248,OK
3,gravil_boug_l900_native_10km_circular.tif,GRAIL,context,9,9,1,10000.00,10000.00,-3.276700e+04,53.3980,122.7969,87.7603,OK
4,lola_downsampled_25m.tif,LOLA,accessibility,3639,3639,1,25.00,25.00,-3.400000e+38,-8111.0000,3929.6001,-1038.3022,OK
5,lola_downsampled_59m.tif,LOLA,accessibility,1516,1516,1,60.00,60.00,-3.400000e+38,-8097.2607,3918.5000,-1039.5721,OK
6,lola_downsampled_236m.tif,LOLA,accessibility,387,387,1,235.00,235.00,-3.400000e+38,-8080.7334,3911.2314,-1045.7619,OK
7,lola_matched_to_lroc_5m_clean.tif,LOLA,accessibility,18195,18195,1,5.00,5.00,-3.400000e+38,-8112.0000,3978.0000,-1037.5748,OK
8,lroc_5m_float32_clean.tif,LROC,context,18195,18195,1,5.00,5.00,-3.400000e+38,0.0166,1.1373,0.0838,OK
9,lroc_shadow_mask_final.tif,LROC,risk,18195,18195,1,5.00,5.00,2.550000e+02,0.0000,1.0000,0.3083,OK
